# APIS Michigan Corpus — Longitudinal Economic Analysis

Deep-dive analysis of the University of Michigan papyrus collection (Karanis, Egypt,
primarily Ptolemaic–Roman periods, c. 250 BCE – 400 CE).

**3,283 translated documents** — the largest single-institution translated corpus.

**Three analytical threads:**
1. **Interest rates** — extracted loan terms; comparison to theoretical risk-free rate
2. **Named-agent tracking** — recurring individuals across documents; longitudinal behavior
3. **Loss aversion signals** — complaint/petition language; reference-point framing

**Primary archive:** Karanis grain transport ostraca (262–312 CE)
documenting individual farmers' deliveries to state granaries — the densest
administrative archive in the collection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
from pathlib import Path
import textwrap

sns.set_theme(style='whitegrid', palette='muted')
ROOT = Path('..').resolve()

# Load all relevant data
apis = pd.read_csv(ROOT / 'processed_data/papyri/apis_combined.csv')
mich = apis[apis['institution'] == 'michigan'].copy()
mich_trans = mich[mich['has_translation'] == True].copy()

rates = pd.read_csv(ROOT / 'processed_data/papyri/interest_rates.csv')
agents_m = pd.read_csv(ROOT / 'processed_data/papyri/michigan_agents.csv')
agents_s = pd.read_csv(ROOT / 'processed_data/papyri/michigan_agents_summary.csv')

print(f"Michigan total:      {len(mich):,}")
print(f"Michigan translated: {len(mich_trans):,}")
print(f"Interest rate obs:   {len(rates)}")
print(f"Agent mentions:      {len(agents_m):,}")
print(f"Unique individuals:  {agents_m.groupby(['name','father']).ngroups}")

## 1. Temporal Distribution of Michigan Corpus

In [ ]:
dated = mich_trans[mich_trans['date_year_approx'].notna()].copy()
print(f"Docs with date: {len(dated):,} ({100*len(dated)/len(mich_trans):.0f}%)")
print(f"Year range: {dated['date_year_approx'].min():.0f} to {dated['date_year_approx'].max():.0f} CE")
print()

# Bin into 50-year periods
bins = range(int(dated['date_year_approx'].min() // 50) * 50,
              int(dated['date_year_approx'].max() // 50) * 50 + 51, 50)
dated['period'] = pd.cut(dated['date_year_approx'], bins=list(bins))

fig, ax = plt.subplots(figsize=(13, 4))
period_counts = dated.groupby('period', observed=True).size()
period_counts.plot(kind='bar', ax=ax, color='#2980b9', edgecolor='white')
ax.set_xlabel('Period (CE)')
ax.set_ylabel('Documents')
ax.set_title('Michigan Papyri — Temporal Distribution (50-year bins)', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(ROOT / 'outputs/michigan_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

# Peak periods
print("Top 5 periods by document count:")
print(period_counts.sort_values(ascending=False).head(5).to_string())

## 2. Genre Breakdown and Economic Content

In [ ]:
# Classify genres into broad categories
def classify_genre(g):
    g = str(g).lower()
    if 'ostracon' in g: return 'ostracon'
    if 'letter' in g: return 'letter'
    if 'petition' in g: return 'petition'
    if 'literary' in g: return 'literary'
    if 'tablet' in g: return 'tablet'
    return 'other_documentary'

mich_trans['genre_cat'] = mich_trans['genre'].apply(classify_genre)

# Economic signal by genre
eco_terms = r'tax|payment|paid|receipt|grain|wheat|barley|drachm|artab|silver|gold|loan|debt|interest'
mich_trans['has_economic'] = mich_trans['translation_text'].str.contains(eco_terms, case=False, na=False)

genre_eco = mich_trans.groupby('genre_cat').agg(
    n=('doc_id','count'),
    n_economic=('has_economic','sum')
).reset_index()
genre_eco['pct_economic'] = 100 * genre_eco['n_economic'] / genre_eco['n']
genre_eco = genre_eco.sort_values('n', ascending=False)

print(genre_eco.to_string(index=False))
print()
print(f"Overall economic content rate: {100*mich_trans['has_economic'].mean():.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(genre_eco['genre_cat'], genre_eco['n'], color='#2980b9')
axes[0].set_title('Document Count by Genre', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(genre_eco['genre_cat'], genre_eco['pct_economic'], color='#27ae60')
axes[1].set_ylabel('% with economic content')
axes[1].set_title('Economic Content Rate by Genre', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(ROOT / 'outputs/michigan_genre_economic.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Interest Rate Data

6 cleanly extracted loan interest rates from APIS Michigan and partner institutions.
The dominant rate of **12% per annum** (1 drachma per mina per month) is the
standard Ptolemaic–Roman legal interest rate — effectively a price-administered rate.

**Behavioral relevance:** A uniform administered rate implies no market clearing through
price adjustment. Rationing must occur through non-price mechanisms — relationship
lending, collateral, or credit rationing. This is consistent with models of credit
rationing under asymmetric information (Stiglitz–Weiss 1981).

In [ ]:
print("Extracted interest rates:")
print()
display_cols = ['date_text','institution','doc_id','rate_type','rate_annual_pct','rate_raw']
print(rates[display_cols].to_string(index=False))
print()
print(f"Modal rate: {rates['rate_annual_pct'].mode()[0]:.0f}% per annum")
print(f"Range:      {rates['rate_annual_pct'].min():.0f}%–{rates['rate_annual_pct'].max():.0f}% per annum")
print()
print("Interpretation:")
print("  1 dr/mina/month = 1%/month = 12%/year (standard Ptolemaic-Roman legal rate)")
print("  2 dr/mina/month = 2%/month = 24%/year (attested 172 BCE; above legal rate)")
print()

# Show the high-rate loan in detail
high = rates[rates['rate_annual_pct'] > 12].iloc[0]
print(f"High-rate loan [{high['doc_id']}] ({high['date_text']}):")
loan_doc = apis[apis['doc_id'] == high['doc_id']].iloc[0]
for line in textwrap.wrap(str(loan_doc['translation_text'])[:600], 100):
    print(f'  {line}')

In [ ]:
# Show the broader interest-rate context landscape
# All docs mentioning interest — categorize mentions
interest_docs = apis[
    (apis['has_translation'] == True) &
    apis['translation_text'].str.contains('interest', case=False, na=False)
].copy()

int_type = {
    'simple interest':    r'simple interest',
    'compound interest':  r'compound interest',
    'rate specified':     r'\d+\s*(?:drachm\w+|obol\w*|%)\s+(?:per|a)\s+(?:mina|month)',
    'interest in default':r'interest\s+(?:for|during)\s+(?:the\s+)?(?:period|delay|default|overtime)',
    'repay with interest':r'repay.{0,20}interest|interest.{0,20}repay',
    'list of interest amounts': r'interest\s+\d+\s+(?:dr|drachm)',
}

print(f"Docs mentioning interest across all APIS institutions: {len(interest_docs)}")
print()
for label, pat in int_type.items():
    n = interest_docs['translation_text'].str.contains(pat, case=False, na=False).sum()
    print(f"  {label:35s}: {n:3d} ({100*n/len(interest_docs):.0f}%)")

## 4. Named-Agent Longitudinal Tracking — Karanis Grain Transporters

The Karanis ostraca (Michigan) are the richest longitudinal archive:
grain transport receipts recording individual farmers' deliveries by name,
often spanning multiple years. This allows reconstruction of individual
economic activity over time — rare in the ancient world.

In [ ]:
print("Michigan named agents (≥2 documents):")
recurring = agents_s[(agents_s['n_docs'] >= 2) & (agents_s['institution'] == 'michigan')]
print(f"  {len(recurring)} individuals appear in ≥2 documents")
print()

top = recurring.sort_values('n_docs', ascending=False).head(20)
for _, r in top.iterrows():
    span = ''
    if pd.notna(r['earliest']) and pd.notna(r['latest']):
        if r['earliest'] < 0:
            span = f"{abs(r['earliest']):.0f}–{abs(r['latest']):.0f} BCE"
        else:
            span = f"{r['earliest']:.0f}–{r['latest']:.0f} CE"
    role = f" [{r['roles']}]" if r['roles'] else ''
    print(f"  {r['name']:18s} s/o {r['father']:18s}  {r['n_docs']:2d} docs  {span}{role}")

In [ ]:
# Focus on Palemon son of Ptollas — best-documented individual
palemon_docs = agents_m[
    (agents_m['name'] == 'Palemon') & (agents_m['father'] == 'Ptollas')
]['doc_id'].unique()

pal_data = apis[apis['doc_id'].isin(palemon_docs)].sort_values('date_year_approx')
print(f"Palemon son of Ptollas — {len(pal_data)} documents")
print(f"Date range: {pal_data['date_year_approx'].min():.0f}–{pal_data['date_year_approx'].max():.0f} CE")
print()

for _, row in pal_data.iterrows():
    print(f"  [{row['doc_id']}] {row['date_text']}")
    print(f"    {str(row['summary'])[:100]}")
    # Extract quantity transported
    text = str(row['translation_text'])
    qty_m = re.search(r'(\d+)\s+donkey(?:s)?\s+(?:of\s+)?(wheat|lentil|barley)', text, re.IGNORECASE)
    if qty_m:
        print(f"    Quantity: {qty_m.group(1)} donkeys of {qty_m.group(2)}")
    print()

In [ ]:
# Visualize Palemon's transport activity over time
pal_records = []
for _, row in pal_data.iterrows():
    text = str(row['translation_text'])
    for m in re.finditer(r'(\d+)\s+donkey(?:s)?\s+(?:of\s+)?(wheat|lentil|barley|grain)', text, re.IGNORECASE):
        pal_records.append({
            'doc_id': row['doc_id'],
            'date_year': row['date_year_approx'],
            'date_text': str(row['date_text'])[:30],
            'qty_donkeys': int(m.group(1)),
            'commodity': m.group(2).lower()
        })

pal_df = pd.DataFrame(pal_records)
print("Palemon's deliveries:")
print(pal_df.to_string(index=False))
print()
print(f"Total donkey-loads: {pal_df['qty_donkeys'].sum()}")
print(f"Average per delivery event: {pal_df['qty_donkeys'].mean():.1f} donkeys")

## 5. Complaint and Loss-Framing Documents

Michigan petitions and letters contain explicit complaint language —
the strongest loss-aversion signal available. These documents frame
economic grievances in terms of deviation from an expected or prior state.

In [ ]:
# Load the coded Michigan data
coded = pd.read_csv(ROOT / 'processed_data/papyri/apis_combined_coded.csv') \
          if (ROOT / 'processed_data/papyri/apis_combined_coded.csv').exists() \
          else None

# If coded file not available, compute signals on the fly
if coded is None:
    print("Coded CSV not found — computing signals inline")
    import sys
    sys.path.insert(0, str(ROOT / 'scripts'))

# Manual complaint/loss pattern on Michigan
complaint_pattern = re.compile(
    r'(?:complaint|petition|aggrieved|wrong|injustice|unjust|damaged|suffer|loss|'
    r'plundered|stolen|deprived|dispossessed|appeal|beg|implore|request)',
    re.IGNORECASE
)
loss_pattern = re.compile(
    r'(?:loss(?:es)?|losing|lost|damage|destroyed|taken|seized|unfairly|wrongly)',
    re.IGNORECASE
)
ref_point_pattern = re.compile(
    r'(?:as before|as agreed|as promised|previously|used to|formerly|customary|'
    r'as was done|as is customary|which was owed)',
    re.IGNORECASE
)

mich_trans['be_complaint'] = mich_trans['translation_text'].str.contains(complaint_pattern, na=False)
mich_trans['be_loss']      = mich_trans['translation_text'].str.contains(loss_pattern, na=False)
mich_trans['be_refpoint']  = mich_trans['translation_text'].str.contains(ref_point_pattern, na=False)
mich_trans['loss_signal']  = mich_trans['be_complaint'] | mich_trans['be_loss']

print(f"Michigan loss signal docs: {mich_trans['loss_signal'].sum()} ({100*mich_trans['loss_signal'].mean():.1f}%)")
print(f"Complaint:                 {mich_trans['be_complaint'].sum()}")
print(f"Loss framing:              {mich_trans['be_loss'].sum()}")
print(f"Reference point:           {mich_trans['be_refpoint'].sum()}")
print(f"All three:                 {(mich_trans['be_complaint'] & mich_trans['be_loss'] & mich_trans['be_refpoint']).sum()}")
print()

# By genre
genre_loss = mich_trans.groupby('genre_cat')['loss_signal'].mean() * 100
print("Loss signal rate by genre (%):") 
print(genre_loss.sort_values(ascending=False).round(1).to_string())

In [ ]:
# Show highest-signal documents: complaint + loss + reference point
high_signal = mich_trans[
    mich_trans['be_complaint'] & mich_trans['be_loss'] & mich_trans['be_refpoint']
].copy()

print(f"High-signal docs (complaint + loss + reference point): {len(high_signal)}")
print()

for _, row in high_signal.head(4).iterrows():
    print(f"{'='*70}")
    print(f"[{row['doc_id']}] {row['date_text']}  ({row['genre_cat']})")
    print('='*70)
    for line in textwrap.wrap(str(row['translation_text'])[:600], 100):
        print(line)
    print()

## 6. Temporal Pattern of Loss Signals

Does the rate of loss-framing language vary over time? If agents respond to
economic conditions, we expect more complaint/loss language during periods
of documented crisis (e.g., the 3rd-century CE economic crisis, the Roman
tax reforms under Diocletian c. 284 CE).

In [ ]:
dated_loss = mich_trans[mich_trans['date_year_approx'].notna()].copy()
# 50-year bins
dated_loss['century'] = (dated_loss['date_year_approx'] // 50 * 50).astype(int)

period_loss = dated_loss.groupby('century').agg(
    n=('doc_id','count'),
    n_loss=('loss_signal','sum'),
    n_complaint=('be_complaint','sum'),
    n_refpoint=('be_refpoint','sum'),
).reset_index()
period_loss['loss_pct'] = 100 * period_loss['n_loss'] / period_loss['n']
period_loss = period_loss[period_loss['n'] >= 10]  # require min sample

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax = axes[0]
ax.bar(period_loss['century'], period_loss['n'], color='#2980b9', alpha=0.7)
ax.set_ylabel('Documents')
ax.set_title('Michigan Corpus: Document Volume Over Time', fontweight='bold')

ax2 = axes[1]
ax2.plot(period_loss['century'], period_loss['loss_pct'], 'ro-', linewidth=1.5, markersize=6)
ax2.axvline(284, color='gray', linestyle='--', alpha=0.5, label='Diocletian reforms (284 CE)')
ax2.axvline(-30, color='green', linestyle='--', alpha=0.5, label='Roman annexation of Egypt (30 BCE)')
ax2.set_ylabel('% documents with loss signal')
ax2.set_xlabel('Year CE (50-year bins)')
ax2.set_title('Loss-Framing Signal Rate Over Time', fontweight='bold')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(ROOT / 'outputs/michigan_loss_signal_time.png', dpi=150, bbox_inches='tight')
plt.show()

print(period_loss[['century','n','loss_pct']].to_string(index=False))

## 7. Summary: Behavioral Economics Findings

### Interest Rates
The dominant rate of 12%/yr (= 1 dr/mina/month) is documented from at least 141 CE
through 454 CE — over 300 years of stable administered pricing. The single above-rate
loan (24%/yr, 172 BCE) predates Roman rule and may reflect pre-standardization
market rates. The administered rate is consistent with a policy-constrained market
where non-price factors govern credit allocation.

### Named Agents
The Karanis grain transport ostraca (262–312 CE) provide the densest longitudinal
individual-level dataset. Palemon son of Ptollas appears in 8 receipts spanning
c. 298–304 CE — a transport obligation of state grain assessed against him annually.
This is an administered quantity (tax in kind) rather than a market transaction:
behavioral variation must appear in effort, timing, and compliance rather than
negotiated quantity.

### Loss Aversion
The petition genre shows the highest loss-signal rate, as expected: petitions are
by design documents in which the petitioner constructs a narrative of wrongful loss
to justify state intervention. The reference-point framing ("as before," "as is
customary") appears in ~X% of translated documents — consistent with behavioral
models in which agents evaluate outcomes relative to a status quo anchor rather
than absolute outcomes.

### Next Steps
- Link individual agent activity to behavioral signals in their texts
- Extract grain quantities from Karanis ostraca for compliance/effort analysis
- Cross-reference Palemon's transport volumes with known crop year quality data